# 04. Dữ liệu ngoài: video, Roboflow/Kaggle, lọc trùng với VNLP

Dùng để tạo **tập kiểm thử thực tế** khi không tự chụp được ảnh. Ba phần độc lập, chạy phần nào cần:

1. Trích khung hình từ video giao thông tải về (chỉ dùng video có giấy phép cho phép, ví dụ Creative Commons)
2. Nhập bộ dữ liệu YOLO tải về (Roboflow, CVAT...) vào cấu trúc của project
3. Lọc ảnh ngoài **trùng với VNLP** (nhiều bộ trên mạng là bản tải lại của VNLP); cần chạy notebook 02 trước

Dữ liệu ngoài để ở `external_dir` (ổ E:), không để trong project.
Ghi nguồn, giấy phép và ngày tải của từng bộ vào `data/external/README.md` (trong project).

In [ ]:
import sys, csv, json, hashlib, collections, os, shutil, random
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))
from alpr.paths import load_paths

PATHS = load_paths()                  # khai báo trong config.json (xem config.example.json)
VNLP = PATHS["vnlp_root"]             # dữ liệu gốc (ổ E:)
DATASETS = PATHS["datasets_dir"]      # dataset sinh ra (để cùng ổ với VNLP cho hard link)
EXTERNAL = PATHS["external_dir"]      # video, bộ dữ liệu ngoài
manifest = pd.read_csv(ROOT / "data/labels/vnlp_manifest.csv", dtype={"label": str, "province": str}).fillna("")
print(len(manifest), "dòng manifest")

In [ ]:
def dhash(img: Image.Image) -> int:
    """Hash 64 bit theo chênh lệch độ sáng; ảnh gần giống nhau có hash gần nhau."""
    g = np.asarray(img.convert("L").resize((9, 8), Image.BILINEAR), dtype=np.int16)
    return int("".join("1" if b else "0" for b in (g[:, 1:] > g[:, :-1]).flatten()), 2)

def hamming(a: int, b: int) -> int:
    return bin(a ^ b).count("1")

## 1. Trích khung hình từ video

Bỏ video vào `<external_dir>/videos/` (mặc định `E:/vietnamese-license-plate-recognition/external/videos`). Mỗi `EVERY_SEC` giây lấy 1 khung hình, bỏ khung gần giống khung trước.
Tên tệp giữ tên video để sau này **chia tập theo video**.

Cần: `pip install opencv-python`

In [ ]:
import cv2

VIDEO_DIR = EXTERNAL / "videos"
FRAME_DIR = EXTERNAL / "frames"
EVERY_SEC = 1.0
MIN_DIFF = 6      # Hamming tối thiểu so với khung đã lưu trước

FRAME_DIR.mkdir(parents=True, exist_ok=True)
videos = [p for p in VIDEO_DIR.rglob("*") if p.suffix.lower() in {".mp4", ".mov", ".avi", ".mkv"}]
print(len(videos), "video")
for v in videos:
    cap = cv2.VideoCapture(str(v))
    step = max(1, int(round((cap.get(cv2.CAP_PROP_FPS) or 25) * EVERY_SEC)))
    idx = saved = 0; last = None
    while True:
        ok, frame = cap.read()
        if not ok:
            break
        if idx % step == 0:
            h = dhash(Image.fromarray(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)))
            if last is None or hamming(h, last) >= MIN_DIFF:
                cv2.imwrite(str(FRAME_DIR / f"{v.stem}__{idx:06d}.jpg"), frame, [cv2.IMWRITE_JPEG_QUALITY, 95])
                last, saved = h, saved + 1
        idx += 1
    cap.release()
    print(f"{v.name}: {saved} khung hình")

Sau bước này: tải thư mục `frames/` lên Roboflow (gói miễn phí) hoặc CVAT, vẽ hộp biển số
(lớp `plate_1row` / `plate_2row`) và ghi chuỗi biển. Xuất về dạng **YOLOv8** rồi dùng phần 2.
Khi đã có detector huấn luyện trên VNLP, nên cho nó gán nhãn trước rồi người sửa lại.

## 2. Nhập bộ dữ liệu YOLO tải về

Thư mục nguồn cần có `data.yaml` và các thư mục `images/`, `labels/` (cấu trúc xuất YOLO thông thường).
`CLASS_MAP` ánh xạ tên lớp của bộ ngoài sang: `0` = biển 1 dòng, `1` = biển 2 dòng, `-1` = bỏ.

In [ ]:
import yaml

SRC = EXTERNAL / "ten_bo_du_lieu"               # sửa
NAME = "ten_bo_du_lieu"                         # sửa
CLASS_MAP = {"License-Plate-VN": 0}             # sửa theo data.yaml của bộ nguồn
SPLIT = "test"                                  # mặc định làm tập kiểm thử

if SRC.exists():
    names = yaml.safe_load((SRC / "data.yaml").read_text(encoding="utf-8"))["names"]
    names = [names[k] for k in sorted(names)] if isinstance(names, dict) else names
    remap = {i: CLASS_MAP[n] for i, n in enumerate(names) if n in CLASS_MAP}
    dst = DATASETS / "external" / NAME
    rows = []
    for img in SRC.rglob("*"):
        if img.suffix.lower() not in {".jpg", ".jpeg", ".png"} or "images" not in img.parts:
            continue
        lbl = Path(str(img.parent).replace("images", "labels")) / (img.stem + ".txt")
        lines = []
        if lbl.exists():
            for ln in lbl.read_text().splitlines():
                p = ln.split()
                if p and remap.get(int(p[0]), -1) >= 0:
                    lines.append(" ".join([str(remap[int(p[0])])] + p[1:5]))
        out_img = dst / "images" / SPLIT / f"{NAME}__{img.name}"
        out_img.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(img, out_img)
        out_lbl = dst / "labels" / SPLIT / (out_img.stem + ".txt")
        out_lbl.parent.mkdir(parents=True, exist_ok=True)
        out_lbl.write_text("\n".join(lines) + ("\n" if lines else ""))
        rows.append(dict(image=out_img.name, n_plates=len(lines), source=NAME, split=SPLIT))
    pd.DataFrame(rows).to_csv(dst / "manifest.csv", index=False)
    (dst / "data.yaml").write_text(f"path: {dst.resolve().as_posix()}\ntrain: images/train\nval: images/val\n"
                                   "test: images/test\nnames:\n  0: plate_1row\n  1: plate_2row\n")
    print(f"Đã nhập {len(rows)} ảnh vào {dst}")
else:
    print("Chưa có thư mục nguồn:", SRC)

## 3. Lọc ảnh ngoài trùng với VNLP

So dHash của ảnh ngoài với dHash của toàn bộ ảnh VNLP (lấy từ `reports/image_check.csv` của notebook 02).
Ảnh nào gần trùng thì **loại khỏi tập kiểm thử**, nếu không kết quả "thực tế" sẽ bị thổi phồng.

In [ ]:
EXT_IMAGES = DATASETS / "external" / NAME / "images"   # hoặc FRAME_DIR
MAX_HAMMING = 4

ic = pd.read_csv(ROOT / "reports/image_check.csv").fillna("")
ref = [(p, int(h, 16)) for p, h in zip(ic.rel_path, ic.dhash) if h]
bands = collections.defaultdict(list)
for k, (_, h) in enumerate(ref):
    for b in range(4):
        bands[(b, (h >> 16 * b) & 0xFFFF)].append(k)

hits, n = [], 0
for img in EXT_IMAGES.rglob("*"):
    if img.suffix.lower() not in {".jpg", ".jpeg", ".png"}:
        continue
    n += 1
    with Image.open(img) as im:
        h = dhash(im)
    cand = {k for b in range(4) for k in bands.get((b, (h >> 16 * b) & 0xFFFF), [])}
    best = min(((hamming(h, ref[k][1]), ref[k][0]) for k in cand), default=None)
    if best and best[0] <= MAX_HAMMING:
        hits.append(dict(external=str(img), vnlp=best[1], hamming=best[0]))
hits = pd.DataFrame(hits)
print(f"{len(hits)}/{n} ảnh ngoài gần trùng với VNLP")
hits.head()

In [ ]:
# Xóa các ảnh trùng khỏi bộ ngoài (bỏ dấu # để chạy)
# for p in hits.external:
#     Path(p).unlink(missing_ok=True)
#     Path(p.replace("images", "labels")).with_suffix(".txt").unlink(missing_ok=True)